In [3]:
# ============================================================
# HVLT FOR READ DATASET (PAGE XML LINE-LEVEL RECOGNITION)
# FULL SINGLE-CELL TRAINING AND EVALUATION NOTEBOOK
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import time
import random
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Training/page"
VAL_DIR   = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Validation/page"

IMG_HEIGHT = 32
IMG_WIDTH  = 192  # Increased for full text lines instead of single words

BATCH_SIZE = 16
LR = 5e-5
MAX_EPOCHS = 100
MAX_SEQ_LEN = 50   # Expanded to fit full textual sentence lines safely
PATIENCE = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True

# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# READ DATASET VOCABULARY
# ============================================================
# Standard multilingual historical support characters
READ_CHARS = (
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "äöüÄÖÜß" 
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*=\"„“¬ "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(READ_CHARS)

char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

# ============================================================
# PAGE XML PARSING & ON-THE-FLY LINE EXTRACTOR
# ============================================================
def parse_page_xml_dataset(directory_path):
    """
    Parses full page images and crops individual lines out of polygon coordinates 
    defined in the companion PAGE XML sheets.
    """
    extracted_samples = []
    
    if not os.path.exists(directory_path):
        print(f"Warning: Path does not exist -> {directory_path}")
        return extracted_samples

    # Automatically extract all XML descriptors in the folder
    all_files = os.listdir(directory_path)
    xml_files = [f for f in all_files if f.endswith('.xml')]
    
    # PAGE XML Namespace map declaration
    ns = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

    for xml_file in tqdm(xml_files, desc=f"Parsing XMLs in {os.path.basename(directory_path)}"):
        xml_path = os.path.join(directory_path, xml_file)
        
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            
            # Find the Page tag elements to extract the true image path filename
            page_elem = root.find('.//ns:Page', ns)
            if page_elem is None:
                continue
                
            img_filename = page_elem.get('imageFilename')
            img_path = os.path.join(directory_path, img_filename)
            
            # Ensure the full raw page image exists before parsing lines
            if not os.path.exists(img_path):
                continue

            # Read the master image once per file to save performance resources
            master_img = cv2.imread(img_path)
            if master_img is None:
                continue
                
            img_h, img_w = master_img.shape[:2]

            # Iterate through every TextLine block item
            for text_line in root.findall('.//ns:TextLine', ns):
                # 1. Parse coordinate points
                coords_elem = text_line.find('ns:Coords', ns)
                if coords_elem is None:
                    continue
                points_str = coords_elem.get('points') # format: "x1,y1 x2,y2 ..."
                
                # 2. Parse text transcriptions safely
                unicode_elem = text_line.find('.//ns:Unicode', ns)
                if unicode_elem is None or unicode_elem.text is None:
                    continue
                text_content = unicode_elem.text.strip()
                if not text_content:
                    continue
                
                # Convert coordinate string into a standard bounding array slice box
                try:
                    pts = [list(map(int, pt.split(','))) for pt in points_str.split(' ')]
                    pts = np.array(pts, dtype=np.int32)
                    
                    # Compute strict horizontal and vertical bounding boundaries
                    x, y, w, h = cv2.boundingRect(pts)
                    
                    # Guard bounding conditions to catch bad polygon data lines safely
                    if w < 5 or h < 5 or x < 0 or y < 0 or (x + w) > img_w or (y + h) > img_h:
                        continue
                        
                    # Extract the unique cropped text line out of the master image directly
                    line_crop = master_img[y:y+h, x:x+w]
                    
                    # Convert to RGB color channel configuration space
                    line_crop = cv2.cvtColor(line_crop, cv2.COLOR_BGR2RGB)
                    text_content = unicodedata.normalize("NFC", text_content)
                    
                    extracted_samples.append((line_crop, text_content))
                except Exception:
                    continue # Bypasses erratic line configurations safely
                    
        except Exception as e:
            print(f"Skipping unreadable XML format sheet {xml_file}: {e}")
            continue

    return extracted_samples

# Load and separate data structures instantly
print("Initializing READ dataset structural extraction pipeline...")
train_samples = parse_page_xml_dataset(TRAIN_DIR)
val_samples   = parse_page_xml_dataset(VAL_DIR)

# Split Validation split evenly to establish an explicit, isolated evaluation test set
val_samples, test_samples = train_test_split(
    val_samples, test_size=0.5, random_state=SEED
)

print(f"\nSuccessfully Parsed Line Data Breakdown Summary:")
print(f"TRAIN LINE IMAGES : {len(train_samples)}")
print(f"VAL LINE IMAGES   : {len(val_samples)}")
print(f"TEST LINE IMAGES  : {len(test_samples)}")

if len(train_samples) == 0 or len(val_samples) == 0:
    raise ValueError("Pipeline processing halted. Verify database directory targets contain matching image pairs.")

# ============================================================
# IMAGE PREPROCESSING ENGINE WITH DATA AUGMENTATION
# ============================================================
def preprocess_cropped_image(cv2_img, augment=False):
    # Apply standard robust data augmentations during training runs
    if augment:
        if random.random() < 0.5:
            alpha = random.uniform(0.8, 1.2)
            beta = random.randint(-15, 15)
            cv2_img = cv2.convertScaleAbs(cv2_img, alpha=alpha, beta=beta)
        if random.random() < 0.3:
            k = random.choice([3, 5])
            cv2_img = cv2.GaussianBlur(cv2_img, (k, k), 0)

    h, w = cv2_img.shape[:2]
    scale = IMG_HEIGHT / h
    new_w = max(1, int(w * scale))

    img = cv2.resize(cv2_img, (new_w, IMG_HEIGHT))

    # Strict padding to ensure uniform tensor shapes
    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))

    return torch.tensor(img, dtype=torch.float32)

# ============================================================
# TEXT TRANSFORMATION UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX

    return torch.tensor(tokens, dtype=torch.long)

def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)

# ============================================================
# MEMORY DATASET MANAGEMENT
# ============================================================
class READLineDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cv2_img, text = self.samples[idx]
        image_tensor = preprocess_cropped_image(cv2_img, augment=self.augment)
        tokens = encode_text(text)
        return image_tensor, tokens, text

train_loader = DataLoader(
    READLineDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    READLineDataset(val_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    READLineDataset(test_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================
class PositionalBridge(nn.Module):
    def __init__(self, in_channels=1024, d_model=768, vis_seq_len=256):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed

# ============================================================
# DECODER
# ============================================================
class READDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=6):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed = nn.Embedding(MAX_SEQ_LEN, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=3072, dropout=0.1, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Aligning Pretrained Multilingual Token Embedding Weights...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            n = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n]
            del xlm
            print("Successfully matched weight layers.")
        except Exception as e:
            print("Pretrained base initializing skipped:", e)

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x = self.token_embed(tgt_tokens) + self.pos_embed(pos)

        tgt_mask = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(
            tgt=x, memory=memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.output_proj(out)

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished = torch.zeros(B, dtype=torch.bool, device=memory.device)

        for _ in range(max_len):
            logits = self.forward(memory, generated)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= (next_token == EOS_IDX)

            if finished.all():
                break
        return generated[:, 1:]

# ============================================================
# CORE ARCHITECTURE ENGINE (HVLT Wrapper)
# ============================================================
class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True, features_only=True,
            img_size=(32, 192), strict_img_size=False,
        )
        self.bridge = PositionalBridge(in_channels=1024, d_model=768, vis_seq_len=256)
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        feats = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens)

    @torch.no_grad()
    def predict(self, images):
        feats = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder.greedy_decode(memory)

# ============================================================
# CRITERIA STRATIFIED UTIL FUNCTIONS
# ============================================================
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)

# ============================================================
# MODEL ALLOCATION
# ============================================================
model = HVLT().to(DEVICE)
optimizer = Adam(model.parameters(), lr=LR)
scaler = GradScaler("cuda", enabled=USE_AMP)

print("TOTAL REGISTERED PARAMS:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

# ============================================================
# TRAINING SYSTEM LOOP CONTROL
# ============================================================
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # --- TRAINING STAGE ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []
    
    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input = targets[:, :-1]
        decoder_target = targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():
            pred_ids = logits.argmax(dim=-1)
            preds = [decode_tokens(x) for x in pred_ids]

        train_preds.extend(preds)
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Train Loss: {loss.item():.4f}")

    train_cer = char_accuracy(train_preds, train_labels)
    train_wer = wer(train_labels, train_preds) * 100
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATION STAGE ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input = targets[:, :-1]
            decoder_target = targets[:, 1:]

            logits = model(images, decoder_input)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

            pred_ids = model.predict(images)
            preds = [decode_tokens(x) for x in pred_ids]

            val_preds.extend(preds)
            val_labels.extend(labels)

    val_cer = char_accuracy(val_preds, val_labels)
    val_wer = wer(val_labels, val_preds) * 100
    avg_val_loss = val_loss / len(val_loader)

    # --- METRICS SYSTEM REPORT ---
    print("\n" + "="*60)
    print(f"EPOCH {epoch}")
    print(f"TRAIN LOSS: {avg_train_loss:.4f} | VAL LOSS: {avg_val_loss:.4f}")
    print(f"TRAIN CAR:  {train_cer:.2f}%  | VAL CAR:  {val_cer:.2f}%")
    print(f"TRAIN WER:  {train_wer:.2f}%  | VAL WER:  {val_wer:.2f}%")
    print("="*60)

    # --- ROBUST STABLE LOSS-BASED EARLY STOPPING ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
        }, "best_read_hvlt.pt")
        print(">>> PERFORMANCE INCREASE DETECTED: VAL LOSS DROPPED. SAVE WEIGHTS.")
    else:
        patience_counter += 1
        print(f">>> No validation step drops. Early stopping matrix check: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOPPING HALT] Training sequence terminated across patience saturation.")
        break

# ============================================================
# FINAL DATA COMPUTE EVALUATION RUN
# ============================================================
print("\nLOADING OPTIMIZED CHECKPOINT MARKS FOR FINAL SYSTEM GENERALIZATION EVALUATION...")
if os.path.exists("best_read_hvlt.pth"):
    checkpoint = torch.load("best_read_hvlt.pth")
    model.load_state_dict(checkpoint["model"])

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing Assessment"):
        images = images.to(DEVICE)
        pred_ids = model.predict(images)
        preds = [decode_tokens(x) for x in pred_ids]
        test_preds.extend(preds)
        test_labels.extend(labels)

test_cer = char_accuracy(test_preds, test_labels)
test_wer = wer(test_labels, test_preds) * 100

print("\n" + "#"*40 + "\nFINAL READ EVALUATION MATRIX TEST METRICS")
print(f"TEST CAR (Character Accuracy Rate) : {test_cer:.2f}%")
print(f"TEST WER (Word Error Rate)       : {test_wer:.2f}%")
print("#"*40)

# ============================================================
# OUTPUT PREVIEW ANALYSIS VISUAL INSPECTIONS
# ============================================================
print("\nSAMPLE ANALYSIS PREVIEW DISPLAYS:\n")
for i in range(min(15, len(test_labels))):
    print(f"[{i+1:02d}] TRUE LINE TEXT : {test_labels[i]}")
    print(f"     PREDICTED OUT  : {test_preds[i]}")
    print("-" * 50)

VOCAB SIZE: 103
Initializing READ dataset structural extraction pipeline...


Parsing XMLs in page: 100%|██████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 29.65it/s]



Successfully Parsed Line Data Breakdown Summary:
TRAIN LINE IMAGES : 8366
VAL LINE IMAGES   : 520
TEST LINE IMAGES  : 520


/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:34

Aligning Pretrained Multilingual Token Embedding Weights...


...


EPOCH 27
TRAIN LOSS: 0.0631 | VAL LOSS: 1.1396
TRAIN CAR:  62.45%  | VAL CAR:  30.40%
TRAIN WER:  26.90%  | VAL WER:  75.49%
>>> No validation step drops. Early stopping matrix check: 19/20


Epoch 28 | Train Loss: 0.0398: 100%|███████████████████████████████████████████████████████████| 523/523 [01:04<00:00,  8.15it/s]
Validating: 100%|████████████████████████████████████████████████████████████████████████████████| 33/33 [00:07<00:00,  4.28it/s]



EPOCH 28
TRAIN LOSS: 0.0588 | VAL LOSS: 1.1658
TRAIN CAR:  62.41%  | VAL CAR:  29.37%
TRAIN WER:  26.30%  | VAL WER:  76.52%
>>> No validation step drops. Early stopping matrix check: 20/20

[EARLY STOPPING HALT] Training sequence terminated across patience saturation.

LOADING OPTIMIZED CHECKPOINT MARKS FOR FINAL SYSTEM GENERALIZATION EVALUATION...


Testing Assessment: 100%|████████████████████████████████████████████████████████████████████████| 33/33 [00:06<00:00,  4.73it/s]


########################################
FINAL READ EVALUATION MATRIX TEST METRICS
TEST CAR (Character Accuracy Rate) : 29.15%
TEST WER (Word Error Rate)       : 75.14%
########################################

SAMPLE ANALYSIS PREVIEW DISPLAYS:

[01] TRUE LINE TEXT : Im Andern Mandat.
     PREDICTED OUT  : Indrendarn Wilder
--------------------------------------------------
[02] TRUE LINE TEXT : Pitet vmb die Inwonūng
     PREDICTED OUT  : Pats vmb die Inwon<UNK>ng
--------------------------------------------------
[03] TRUE LINE TEXT : Schreiben
     PREDICTED OUT  : Schreiben
--------------------------------------------------
[04] TRUE LINE TEXT : Mūsicant
     PREDICTED OUT  : G<UNK>sicam
--------------------------------------------------
[05] TRUE LINE TEXT : hanns Maximilian Jōrger.
     PREDICTED OUT  : Hanns Wann Jann herrn
--------------------------------------------------
[06] TRUE LINE TEXT : Loblichen Cammer
     PREDICTED OUT  : Callicher Cammer.
--------------------------

## 1

In [6]:
# ============================================================
# HVLT FOR READ DATASET - IMPROVED VERSION
# Changes from original:
#   1. MAX_SEQ_LEN: 50 → 200 (fixes truncation of long lines)
#   2. IMG_HEIGHT/WIDTH: 32x192 → 64x512 (better stroke detail)
#   3. PositionalBridge vis_seq_len: 256 → 1024 (matches new width)
#   4. Stronger augmentation: elastic distortion, morphology, noise
#   5. Beam search decoding instead of greedy
#   6. LR scheduler (CosineAnnealing) added
#   7. Correct checkpoint filename (best_read_hvlt.pt, not .pth)
#   8. char_accuracy fixed to be true CER (not misnamed CAR)
# ============================================================


# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import time
import random
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR   # NEW
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

# Optional: albumentations for stronger augmentation
# pip install albumentations
try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("albumentations not installed. Using basic augmentation only.")
    print("Install with: pip install albumentations")

29.15%
# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Training/page"
VAL_DIR   = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Validation/page"

# FIX 1: Increased resolution — was 32x192, now 64x512
IMG_HEIGHT = 64
IMG_WIDTH  = 512

BATCH_SIZE = 8       # Reduced from 16 because larger images use more memory
LR = 5e-5
MAX_EPOCHS = 100

# FIX 2: Sequence length — was 50, now 200
# Most full text lines in READ are 80-150 characters; 50 caused severe truncation
MAX_SEQ_LEN = 200

PATIENCE = 20
BEAM_WIDTH = 5       # NEW: beam search width for decoding

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True


# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# VOCABULARY
# ============================================================
READ_CHARS = (
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "äöüÄÖÜß"
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*=\"\u201e\u201c\u00ac "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(READ_CHARS)

char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)


# ============================================================
# PAGE XML PARSING
# ============================================================
def parse_page_xml_dataset(directory_path):
    extracted_samples = []

    if not os.path.exists(directory_path):
        print(f"Warning: Path does not exist -> {directory_path}")
        return extracted_samples

    all_files = os.listdir(directory_path)
    xml_files = [f for f in all_files if f.endswith('.xml')]
    ns = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

    for xml_file in tqdm(xml_files, desc=f"Parsing XMLs in {os.path.basename(directory_path)}"):
        xml_path = os.path.join(directory_path, xml_file)

        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            page_elem = root.find('.//ns:Page', ns)
            if page_elem is None:
                continue

            img_filename = page_elem.get('imageFilename')
            img_path = os.path.join(directory_path, img_filename)

            if not os.path.exists(img_path):
                continue

            master_img = cv2.imread(img_path)
            if master_img is None:
                continue

            img_h, img_w = master_img.shape[:2]

            for text_line in root.findall('.//ns:TextLine', ns):
                coords_elem = text_line.find('ns:Coords', ns)
                if coords_elem is None:
                    continue
                points_str = coords_elem.get('points')

                unicode_elem = text_line.find('.//ns:Unicode', ns)
                if unicode_elem is None or unicode_elem.text is None:
                    continue
                text_content = unicode_elem.text.strip()
                if not text_content:
                    continue

                try:
                    pts = [list(map(int, pt.split(','))) for pt in points_str.split(' ')]
                    pts = np.array(pts, dtype=np.int32)

                    x, y, w, h = cv2.boundingRect(pts)

                    if w < 5 or h < 5 or x < 0 or y < 0 or (x + w) > img_w or (y + h) > img_h:
                        continue

                    line_crop = master_img[y:y+h, x:x+w]
                    line_crop = cv2.cvtColor(line_crop, cv2.COLOR_BGR2RGB)
                    text_content = unicodedata.normalize("NFC", text_content)

                    extracted_samples.append((line_crop, text_content))
                except Exception:
                    continue

        except Exception as e:
            print(f"Skipping {xml_file}: {e}")
            continue

    return extracted_samples


print("Initializing READ dataset extraction...")
train_samples = parse_page_xml_dataset(TRAIN_DIR)
val_samples   = parse_page_xml_dataset(VAL_DIR)

val_samples, test_samples = train_test_split(
    val_samples, test_size=0.5, random_state=SEED
)

print(f"\nData breakdown:")
print(f"TRAIN: {len(train_samples)}")
print(f"VAL:   {len(val_samples)}")
print(f"TEST:  {len(test_samples)}")

if len(train_samples) == 0 or len(val_samples) == 0:
    raise ValueError("No data found. Check directory paths.")


# ============================================================
# FIX 3: STRONGER AUGMENTATION
# ============================================================
if HAS_ALBUMENTATIONS:
    # Elastic distortion simulates ink/paper warping seen in manuscripts
    strong_aug = A.Compose([
        A.ElasticTransform(alpha=30, sigma=5, p=0.4),
        A.Rotate(limit=3, border_mode=cv2.BORDER_REPLICATE, p=0.3),
        A.GaussNoise(var_limit=(5, 25), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.Morphological(scale=(1, 2), operation='dilation', p=0.2),
        A.Morphological(scale=(1, 2), operation='erosion',  p=0.2),
    ])
else:
    strong_aug = None


def preprocess_cropped_image(cv2_img, augment=False):
    # Apply augmentation on the raw crop before resizing
    if augment:
        if strong_aug is not None:
            cv2_img = strong_aug(image=cv2_img)["image"]
        else:
            # Fallback basic augmentation
            if random.random() < 0.5:
                alpha = random.uniform(0.8, 1.2)
                beta  = random.randint(-15, 15)
                cv2_img = cv2.convertScaleAbs(cv2_img, alpha=alpha, beta=beta)
            if random.random() < 0.3:
                k = random.choice([3, 5])
                cv2_img = cv2.GaussianBlur(cv2_img, (k, k), 0)

    h, w = cv2_img.shape[:2]
    scale = IMG_HEIGHT / h
    new_w = max(1, int(w * scale))

    img = cv2.resize(cv2_img, (new_w, IMG_HEIGHT))

    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))

    return torch.tensor(img, dtype=torch.float32)


# ============================================================
# TEXT UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX   # Always end with EOS

    return torch.tensor(tokens, dtype=torch.long)


def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)


# ============================================================
# DATASET
# ============================================================
class READLineDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cv2_img, text = self.samples[idx]
        image_tensor = preprocess_cropped_image(cv2_img, augment=self.augment)
        tokens = encode_text(text)
        return image_tensor, tokens, text


train_loader = DataLoader(
    READLineDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    READLineDataset(val_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    READLineDataset(test_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)


# ============================================================
# FIX 4: POSITIONAL BRIDGE — vis_seq_len scaled to new width
# Original: vis_seq_len=256 for 192px width
# New:      vis_seq_len=1024 for 512px width (4x wider → 4x more visual tokens)
# ============================================================
class PositionalBridge(nn.Module):
    def __init__(self, in_channels=1024, d_model=768, vis_seq_len=1024):
        super().__init__()
        self.pool      = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj      = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed


# ============================================================
# DECODER WITH BEAM SEARCH
# ============================================================
class READDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768, n_heads=12, n_layers=6):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed   = nn.Embedding(MAX_SEQ_LEN, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=3072,
            dropout=0.1, batch_first=True
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Aligning Pretrained Multilingual Token Embedding Weights...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            n   = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n]
            del xlm
            print("Successfully matched weight layers.")
        except Exception as e:
            print("Pretrained init skipped:", e)

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x    = self.token_embed(tgt_tokens) + self.pos_embed(pos)

        tgt_mask             = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(
            tgt=x, memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.output_proj(out)

    # ----------------------------------------------------------
    # FIX 5: BEAM SEARCH DECODING
    # Replaces greedy_decode. BEAM_WIDTH=1 reduces to greedy.
    # ----------------------------------------------------------
    @torch.no_grad()
    def beam_decode(self, memory, beam_width=BEAM_WIDTH, max_len=MAX_SEQ_LEN):
        """
        Beam search decoder.
        memory: (B, seq_len, d_model)
        Returns: list of B decoded token tensors
        """
        B = memory.shape[0]
        results = []

        for b in range(B):
            mem = memory[b:b+1]   # (1, seq_len, d_model)

            # Each beam: (score, token_list)
            beams = [(0.0, [SOS_IDX])]
            completed = []

            for _ in range(max_len):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                        continue

                    t     = torch.tensor([tokens], dtype=torch.long, device=memory.device)
                    logit = self.forward(mem, t)
                    log_p = F.log_softmax(logit[0, -1], dim=-1)

                    topk_scores, topk_ids = log_p.topk(beam_width)
                    for s, idx in zip(topk_scores.tolist(), topk_ids.tolist()):
                        candidates.append((score + s, tokens + [idx]))

                if not candidates:
                    break

                # Keep top beam_width beams
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:beam_width]

                # Stop if all beams ended
                if all(t[-1] == EOS_IDX for _, t in beams):
                    completed.extend(beams)
                    break

            if not completed:
                completed = beams

            # Pick the best completed beam
            best_tokens = max(completed, key=lambda x: x[0])[1]
            results.append(torch.tensor(best_tokens[1:], dtype=torch.long))  # strip SOS

        return results

    # Keep greedy as a fast fallback for training-time validation
    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B         = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished  = torch.zeros(B, dtype=torch.bool, device=memory.device)

        for _ in range(max_len):
            logits     = self.forward(memory, generated)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= (next_token == EOS_IDX)

            if finished.all():
                break

        return generated[:, 1:]


# ============================================================
# MODEL
# ============================================================
class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        # img_size must match new IMG_HEIGHT x IMG_WIDTH
        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True, features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH), strict_img_size=False,
        )
        self.bridge  = PositionalBridge(in_channels=1024, d_model=768, vis_seq_len=1024)
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        feats  = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens)

    def _encode(self, images):
        feats  = self.encoder(images)[-1]
        return self.bridge(feats)

    @torch.no_grad()
    def predict(self, images, use_beam=True):
        memory = self._encode(images)
        if use_beam:
            token_lists = self.decoder.beam_decode(memory)
            return token_lists   # list of tensors (variable length per sample)
        else:
            return self.decoder.greedy_decode(memory)   # (B, T) tensor


# ============================================================
# LOSS & METRICS
# ============================================================
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def char_accuracy(preds, labels):
    """
    Character-level accuracy (true CER complement).
    Counts per-position matches up to the shorter string length,
    penalises length mismatch.
    """
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)


def decode_beam_results(beam_results):
    """Convert list of variable-length tensors from beam_decode to strings."""
    return [decode_tokens(t.tolist()) for t in beam_results]


# ============================================================
# MODEL + OPTIMIZER + SCHEDULER
# ============================================================
# ============================================================
# LAYERWISE LEARNING RATE DECAY (LLRD)
# Each Swin stage gets a smaller LR the deeper (earlier) it is.
# Decoder/bridge get full LR, patch embedding gets the smallest.
# ============================================================
LLRD_DECAY = 0.75   # tune between 0.65–0.85; lower = more aggressive decay

def get_llrd_param_groups(model, base_lr, decay):
    """
    Returns optimizer param groups with decayed LRs per layer group.
    Group order (highest LR → lowest LR):
      1. Decoder + bridge         → base_lr × 1.0
      2. Swin stage 4 (deepest)   → base_lr × decay^1
      3. Swin stage 3             → base_lr × decay^2
      4. Swin stage 2             → base_lr × decay^3
      5. Swin stage 1 + patch emb → base_lr × decay^4
    """
    # Collect named parameters into layer groups
    decoder_bridge_params = []
    stage4_params = []
    stage3_params = []
    stage2_params = []
    stage1_embed_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("decoder.") or name.startswith("bridge."):
            decoder_bridge_params.append(param)

        elif "encoder.layers.3" in name or "encoder.norm3" in name:
            # Swin stage 4 — last (deepest feature) stage
            stage4_params.append(param)

        elif "encoder.layers.2" in name or "encoder.norm2" in name:
            stage3_params.append(param)

        elif "encoder.layers.1" in name or "encoder.norm1" in name:
            stage2_params.append(param)

        else:
            # stage 0, patch embed, pos embed, anything else
            stage1_embed_params.append(param)

    param_groups = [
        {"params": decoder_bridge_params, "lr": base_lr},
        {"params": stage4_params,         "lr": base_lr * decay},
        {"params": stage3_params,         "lr": base_lr * decay**2},
        {"params": stage2_params,         "lr": base_lr * decay**3},
        {"params": stage1_embed_params,   "lr": base_lr * decay**4},
    ]

    # Print LR assigned to each group for verification
    names = ["decoder+bridge", "swin stage 4", "swin stage 3",
             "swin stage 2", "swin stage 1+embed"]
    for n, g in zip(names, param_groups):
        count = len(g["params"])
        print(f"  {n:25s}  lr={g['lr']:.2e}  ({count} tensors)")

    return param_groups


print("\nBuilding LLRD param groups:")
param_groups = get_llrd_param_groups(model, LR, LLRD_DECAY)
optimizer    = Adam(param_groups)

# Scheduler still works fine — it scales each group's LR proportionally
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-7)

scaler = GradScaler("cuda", enabled=USE_AMP)

print("TOTAL PARAMS:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


# ============================================================
# TRAINING LOOP
# ============================================================
best_val_loss     = float('inf')
patience_counter  = 0
CHECKPOINT_PATH   = "best_read_hvlt_2.pt"   # FIX 7: consistent filename

for epoch in range(1, MAX_EPOCHS + 1):

    # --- TRAIN ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input = targets[:, :-1]
        decoder_target = targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():
            pred_ids = logits.argmax(dim=-1)
            preds = [decode_tokens(x) for x in pred_ids]

        train_preds.extend(preds)
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    # Step LR scheduler after each epoch
    scheduler.step()

    train_car = char_accuracy(train_preds, train_labels)
    train_wer_score = wer(train_labels, train_preds) * 100
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATE (use greedy for speed during training) ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input  = targets[:, :-1]
            decoder_target = targets[:, 1:]

            logits   = model(images, decoder_input)
            loss     = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

            # Use greedy during training validation (fast)
            pred_ids = model.predict(images, use_beam=False)
            preds    = [decode_tokens(x) for x in pred_ids]

            val_preds.extend(preds)
            val_labels.extend(labels)

    val_car       = char_accuracy(val_preds, val_labels)
    val_wer_score = wer(val_labels, val_preds) * 100
    avg_val_loss  = val_loss / len(val_loader)

    print("\n" + "="*60)
    print(f"EPOCH {epoch}  |  LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"TRAIN LOSS: {avg_train_loss:.4f}  |  VAL LOSS: {avg_val_loss:.4f}")
    print(f"TRAIN CAR:  {train_car:.2f}%      |  VAL CAR:  {val_car:.2f}%")
    print(f"TRAIN WER:  {train_wer_score:.2f}%      |  VAL WER:  {val_wer_score:.2f}%")
    print("="*60)

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save({
            "model":    model.state_dict(),
            "epoch":    epoch,
            "val_loss": avg_val_loss,
        }, CHECKPOINT_PATH)
        print(">>> Checkpoint saved (val loss improved).")
    else:
        patience_counter += 1
        print(f">>> No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOP] Patience exhausted at epoch {epoch}.")
        break


# ============================================================
# FINAL EVALUATION (with beam search)
# ============================================================
print("\nLoading best checkpoint for final evaluation...")
# FIX 7: filename was 'best_read_hvlt.pth' in original — now consistent '.pt'
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing with beam search"):
        images = images.to(DEVICE)
        # FIX 5: Use beam search for final evaluation
        beam_results = model.predict(images, use_beam=True)
        preds        = decode_beam_results(beam_results)
        test_preds.extend(preds)
        test_labels.extend(labels)

test_car       = char_accuracy(test_preds, test_labels)
test_wer_score = wer(test_labels, test_preds) * 100

print("\n" + "#"*40)
print("FINAL TEST METRICS")
print(f"TEST CAR (Character Accuracy Rate) : {test_car:.2f}%")
print(f"TEST WER (Word Error Rate)         : {test_wer_score:.2f}%")
print("#"*40)


# ============================================================
# SAMPLE PREVIEW
# ============================================================
print("\nSAMPLE PREDICTIONS:\n")
for i in range(min(15, len(test_labels))):
    print(f"[{i+1:02d}] TRUE : {test_labels[i]}")
    print(f"     PRED : {test_preds[i]}")
    print("-" * 60)

albumentations not installed. Using basic augmentation only.
Install with: pip install albumentations
VOCAB SIZE: 103
Initializing READ dataset extraction...


Parsing XMLs in page: 100%|██████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 28.26it/s]



Data breakdown:
TRAIN: 8366
VAL:   520
TEST:  520

Building LLRD param groups:
  decoder+bridge             lr=5.00e-05  (115 tensors)
  swin stage 4               lr=3.75e-05  (0 tensors)
  swin stage 3               lr=2.81e-05  (0 tensors)
  swin stage 2               lr=2.11e-05  (0 tensors)
  swin stage 1+embed         lr=1.58e-05  (325 tensors)
TOTAL PARAMS: 145.306591 M


Epoch 1 | Loss: 0.8175: 100%|████████████████████████████████████████████████████████████████| 1046/1046 [02:17<00:00,  7.62it/s]
Validating: 100%|████████████████████████████████████████████████████████████████████████████████| 65/65 [00:22<00:00,  2.90it/s]



EPOCH 1  |  LR: 5.00e-05
TRAIN LOSS: 0.4626  |  VAL LOSS: 0.5088
TRAIN CAR:  55.19%      |  VAL CAR:  41.51%
TRAIN WER:  62.52%      |  VAL WER:  61.92%
>>> Checkpoint saved (val loss improved).


...


EPOCH 23  |  LR: 4.38e-05
TRAIN LOSS: 0.0261  |  VAL LOSS: 0.5120
TRAIN CAR:  63.12%      |  VAL CAR:  48.41%
TRAIN WER:  20.80%      |  VAL WER:  48.11%
>>> No improvement. Patience: 19/20


Epoch 24 | Loss: 0.0147: 100%|███████████████████████████████████████████████████████████████| 1046/1046 [02:18<00:00,  7.53it/s]
Validating: 100%|████████████████████████████████████████████████████████████████████████████████| 65/65 [00:28<00:00,  2.24it/s]



EPOCH 24  |  LR: 4.32e-05
TRAIN LOSS: 0.0258  |  VAL LOSS: 0.5525
TRAIN CAR:  63.00%      |  VAL CAR:  47.46%
TRAIN WER:  20.61%      |  VAL WER:  48.97%
>>> No improvement. Patience: 20/20

[EARLY STOP] Patience exhausted at epoch 24.

Loading best checkpoint for final evaluation...
Loaded checkpoint from epoch 4


Testing with beam search: 100%|██████████████████████████████████████████████████████████████████| 65/65 [15:06<00:00, 13.95s/it]


########################################
FINAL TEST METRICS
TEST CAR (Character Accuracy Rate) : 43.04%
TEST WER (Word Error Rate)         : 53.74%
########################################

SAMPLE PREDICTIONS:

[01] TRUE : Im Andern Mandat.
     PRED : Jandtigen La<UNK>dedt.
------------------------------------------------------------
[02] TRUE : Pitet vmb die Inwonūng
     PRED : Pitet vmb die Inwon<UNK>ng
------------------------------------------------------------
[03] TRUE : Schreiben
     PRED : Schreiben
------------------------------------------------------------
[04] TRUE : Mūsicant
     PRED : Miesenam
------------------------------------------------------------
[05] TRUE : hanns Maximilian Jōrger.
     PRED : Hanns Maimenden J<UNK>rger
------------------------------------------------------------
[06] TRUE : Loblichen Cammer
     PRED : Lablicher Cr<UNK>mer¬
------------------------------------------------------------
[07] TRUE : dergleichen: one habende
     PRED : dergleich

## 2

In [7]:
# ============================================================
# HVLT FOR READ DATASET - IMPROVED VERSION WITH FIXES
# Changes from original:
#   1. MAX_SEQ_LEN: 50 → 200 (fixes truncation of long lines)
#   2. IMG_HEIGHT/WIDTH: 32x192 → 64x512 (better stroke detail)
#   3. PositionalBridge vis_seq_len: 256 → 1024 (matches new width)
#   4. Stronger augmentation: elastic distortion, morphology, noise
#   5. Beam search decoding instead of greedy
#   6. LR scheduler (CosineAnnealing) added
#   7. Correct checkpoint filename (best_read_hvlt.pt, not .pth)
#   8. char_accuracy fixed to be true CER (not misnamed CAR)
#   9. FIX 1: Freeze encoder for first 5 epochs, unfreeze & rebuild opt
#   10. FIX 2: Downscaled backbone to Swin-Small and reduced decoder size
#   11. FIX 3: Increased val split size (test_size=0.3 instead of 0.5)
# ============================================================


# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import time
import random
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

# Optional: albumentations for stronger augmentation
try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("albumentations not installed. Using basic augmentation only.")
    print("Install with: pip install albumentations")


# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Training/page"
VAL_DIR   = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Validation/page"

# Increased resolution — was 32x192, now 64x512
IMG_HEIGHT = 64
IMG_WIDTH  = 512

BATCH_SIZE = 8       # Reduced from 16 because larger images use more memory
LR = 5e-5
MAX_EPOCHS = 100

# Sequence length — was 50, now 200
MAX_SEQ_LEN = 200

PATIENCE = 20
BEAM_WIDTH = 5       # beam search width for decoding

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True

ENCODER_UNFREEZE_EPOCH = 5   # Unfreeze after this epoch


# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# VOCABULARY
# ============================================================
READ_CHARS = (
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "äöüÄÖÜß"
    "0123456789"
    ".,!?()[]{}:;-_'/\\&%$#@+*=\"\u201e\u201c\u00ac "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(READ_CHARS)

char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)


# ============================================================
# PAGE XML PARSING
# ============================================================
def parse_page_xml_dataset(directory_path):
    extracted_samples = []

    if not os.path.exists(directory_path):
        print(f"Warning: Path does not exist -> {directory_path}")
        return extracted_samples

    all_files = os.listdir(directory_path)
    xml_files = [f for f in all_files if f.endswith('.xml')]
    ns = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

    for xml_file in tqdm(xml_files, desc=f"Parsing XMLs in {os.path.basename(directory_path)}"):
        xml_path = os.path.join(directory_path, xml_file)

        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            page_elem = root.find('.//ns:Page', ns)
            if page_elem is None:
                continue

            img_filename = page_elem.get('imageFilename')
            img_path = os.path.join(directory_path, img_filename)

            if not os.path.exists(img_path):
                continue

            master_img = cv2.imread(img_path)
            if master_img is None:
                continue

            img_h, img_w = master_img.shape[:2]

            for text_line in root.findall('.//ns:TextLine', ns):
                coords_elem = text_line.find('ns:Coords', ns)
                if coords_elem is None:
                    continue
                points_str = coords_elem.get('points')

                unicode_elem = text_line.find('.//ns:Unicode', ns)
                if unicode_elem is None or unicode_elem.text is None:
                    continue
                text_content = unicode_elem.text.strip()
                if not text_content:
                    continue

                try:
                    pts = [list(map(int, pt.split(','))) for pt in points_str.split(' ')]
                    pts = np.array(pts, dtype=np.int32)

                    x, y, w, h = cv2.boundingRect(pts)

                    if w < 5 or h < 5 or x < 0 or y < 0 or (x + w) > img_w or (y + h) > img_h:
                        continue

                    line_crop = master_img[y:y+h, x:x+w]
                    line_crop = cv2.cvtColor(line_crop, cv2.COLOR_BGR2RGB)
                    text_content = unicodedata.normalize("NFC", text_content)

                    extracted_samples.append((line_crop, text_content))
                except Exception:
                    continue

        except Exception as e:
            print(f"Skipping {xml_file}: {e}")
            continue

    return extracted_samples


print("Initializing READ dataset extraction...")
train_samples = parse_page_xml_dataset(TRAIN_DIR)
val_samples   = parse_page_xml_dataset(VAL_DIR)

# FIX 3: Changed test_size from 0.5 to 0.3 to increase validation split tracking robustness
val_samples, test_samples = train_test_split(
    val_samples, test_size=0.3, random_state=SEED
)

print(f"\nData breakdown:")
print(f"TRAIN: {len(train_samples)}")
print(f"VAL:   {len(val_samples)}")
print(f"TEST:  {len(test_samples)}")

if len(train_samples) == 0 or len(val_samples) == 0:
    raise ValueError("No data found. Check directory paths.")


# ============================================================
# STRONGER AUGMENTATION
# ============================================================
if HAS_ALBUMENTATIONS:
    strong_aug = A.Compose([
        A.ElasticTransform(alpha=30, sigma=5, p=0.4),
        A.Rotate(limit=3, border_mode=cv2.BORDER_REPLICATE, p=0.3),
        A.GaussNoise(var_limit=(5, 25), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.Morphological(scale=(1, 2), operation='dilation', p=0.2),
        A.Morphological(scale=(1, 2), operation='erosion',  p=0.2),
    ])
else:
    strong_aug = None


def preprocess_cropped_image(cv2_img, augment=False):
    if augment:
        if strong_aug is not None:
            cv2_img = strong_aug(image=cv2_img)["image"]
        else:
            if random.random() < 0.5:
                alpha = random.uniform(0.8, 1.2)
                beta  = random.randint(-15, 15)
                cv2_img = cv2.convertScaleAbs(cv2_img, alpha=alpha, beta=beta)
            if random.random() < 0.3:
                k = random.choice([3, 5])
                cv2_img = cv2.GaussianBlur(cv2_img, (k, k), 0)

    h, w = cv2_img.shape[:2]
    scale = IMG_HEIGHT / h
    new_w = max(1, int(w * scale))

    img = cv2.resize(cv2_img, (new_w, IMG_HEIGHT))

    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))

    return torch.tensor(img, dtype=torch.float32)


# ============================================================
# TEXT UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX

    return torch.tensor(tokens, dtype=torch.long)


def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)


# ============================================================
# DATASET
# ============================================================
class READLineDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cv2_img, text = self.samples[idx]
        image_tensor = preprocess_cropped_image(cv2_img, augment=self.augment)
        tokens = encode_text(text)
        return image_tensor, tokens, text


train_loader = DataLoader(
    READLineDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    READLineDataset(val_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    READLineDataset(test_samples, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)


# ============================================================
# POSITIONAL BRIDGE
# ============================================================
class PositionalBridge(nn.Module):
    # FIX 2: Switched in_channels from 1024 to 768 to align with Swin-Small
    def __init__(self, in_channels=768, d_model=512, vis_seq_len=1024):
        super().__init__()
        self.pool      = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj      = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed


# ============================================================
# DECODER WITH BEAM SEARCH
# ============================================================
class READDecoder(nn.Module):
    # FIX 2: Downscaled dimensions from 768/12/6 to 512/8/4 for better regularization
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed   = nn.Embedding(MAX_SEQ_LEN, d_model)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=2048, # scaled down feedforward proportionally
            dropout=0.1, batch_first=True
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Aligning Pretrained Multilingual Token Embedding Weights...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            
            # Project/Truncate xlm weights to match scaled-down d_model dimension
            n = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n, :d_model]
            del xlm
            print("Successfully matched weight layers.")
        except Exception as e:
            print("Pretrained init skipped:", e)

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x    = self.token_embed(tgt_tokens) + self.pos_embed(pos)

        tgt_mask             = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(
            tgt=x, memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.output_proj(out)

    @torch.no_grad()
    def beam_decode(self, memory, beam_width=BEAM_WIDTH, max_len=MAX_SEQ_LEN):
        B = memory.shape[0]
        results = []

        for b in range(B):
            mem = memory[b:b+1]

            beams = [(0.0, [SOS_IDX])]
            completed = []

            for _ in range(max_len):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                        continue

                    t     = torch.tensor([tokens], dtype=torch.long, device=memory.device)
                    logit = self.forward(mem, t)
                    log_p = F.log_softmax(logit[0, -1], dim=-1)

                    topk_scores, topk_ids = log_p.topk(beam_width)
                    for s, idx in zip(topk_scores.tolist(), topk_ids.tolist()):
                        candidates.append((score + s, tokens + [idx]))

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:beam_width]

                if all(t[-1] == EOS_IDX for _, t in beams):
                    completed.extend(beams)
                    break

            if not completed:
                completed = beams

            best_tokens = max(completed, key=lambda x: x[0])[1]
            results.append(torch.tensor(best_tokens[1:], dtype=torch.long))

        return results

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B         = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished  = torch.zeros(B, dtype=torch.bool, device=memory.device)

        for _ in range(max_len):
            logits     = self.forward(memory, generated)
            next_token = logits[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)

            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished |= (next_token == EOS_IDX)

            if finished.all():
                break

        return generated[:, 1:]


# ============================================================
# MODEL
# ============================================================
class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        # FIX 2: Changed to swin_small (49M params vs 88M params)
        self.encoder = timm.create_model(
            "swin_small_patch4_window7_224",
            pretrained=True, features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH), strict_img_size=False,
        )
        self.bridge  = PositionalBridge(in_channels=768, d_model=512, vis_seq_len=1024)
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        feats  = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens)

    def _encode(self, images):
        feats  = self.encoder(images)[-1]
        return self.bridge(feats)

    @torch.no_grad()
    def predict(self, images, use_beam=True):
        memory = self._encode(images)
        if use_beam:
            return self.decoder.beam_decode(memory)
        else:
            return self.decoder.greedy_decode(memory)


# ============================================================
# INITIALIZE MODEL & AUX UTILS
# ============================================================
model = HVLT().to(DEVICE)

# FIX 1: Helper function to control encoder gradients
def set_encoder_trainable(model, trainable):
    for param in model.encoder.parameters():
        param.requires_grad = trainable
    status = "UNFROZEN" if trainable else "FROZEN"
    print(f">>> Encoder {status}")

# Freeze encoder initially at start
set_encoder_trainable(model, trainable=False)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)


def decode_beam_results(beam_results):
    return [decode_tokens(t.tolist()) for t in beam_results]


# ============================================================
# LAYERWISE LEARNING RATE DECAY (LLRD)
# ============================================================
LLRD_DECAY = 0.75 

def get_llrd_param_groups(model, base_lr, decay):
    decoder_bridge_params = []
    stage4_params = []
    stage3_params = []
    stage2_params = []
    stage1_embed_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("decoder.") or name.startswith("bridge."):
            decoder_bridge_params.append(param)
        elif "encoder.layers.3" in name or "encoder.norm3" in name:
            stage4_params.append(param)
        elif "encoder.layers.2" in name or "encoder.norm2" in name:
            stage3_params.append(param)
        elif "encoder.layers.1" in name or "encoder.norm1" in name:
            stage2_params.append(param)
        else:
            stage1_embed_params.append(param)

    param_groups = [
        {"params": decoder_bridge_params, "lr": base_lr},
        {"params": stage4_params,         "lr": base_lr * decay},
        {"params": stage3_params,         "lr": base_lr * decay**2},
        {"params": stage2_params,         "lr": base_lr * decay**3},
        {"params": stage1_embed_params,   "lr": base_lr * decay**4},
    ]

    names = ["decoder+bridge", "swin stage 4", "swin stage 3", "swin stage 2", "swin stage 1+embed"]
    for n, g in zip(names, param_groups):
        print(f"  {n:25s}  lr={g['lr']:.2e}  ({len(g['params'])} tensors)")

    return param_groups


print("\nBuilding LLRD param groups (Encoder Initially Frozen):")
param_groups = get_llrd_param_groups(model, LR, LLRD_DECAY)
optimizer    = Adam(param_groups)

# Warmup scheduling window for the frozen section
scheduler = CosineAnnealingLR(optimizer, T_max=ENCODER_UNFREEZE_EPOCH, eta_min=1e-7)

scaler = GradScaler("cuda", enabled=USE_AMP)
print("TOTAL PARAMS:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


# ============================================================
# TRAINING LOOP
# ============================================================
best_val_loss     = float('inf')
patience_counter  = 30
CHECKPOINT_PATH   = "best_read_hvlt_3.pt"

for epoch in range(1, MAX_EPOCHS + 1):

    # FIX 1: Unfreeze encoder after setup epochs and rebuild optimization states
    if epoch == ENCODER_UNFREEZE_EPOCH + 1:
        set_encoder_trainable(model, trainable=True)
        # Rebuild optimizer with LLRD now that encoder parameters accept updates
        param_groups = get_llrd_param_groups(model, LR, LLRD_DECAY)
        optimizer    = Adam(param_groups, weight_decay=1e-4)
        scheduler    = CosineAnnealingLR(optimizer, 
                                         T_max=MAX_EPOCHS - ENCODER_UNFREEZE_EPOCH, 
                                         eta_min=1e-7)
        print(">>> Optimizer rebuilt with LLRD for full model training")

    # --- TRAIN ---
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input = targets[:, :-1]
        decoder_target = targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():
            pred_ids = logits.argmax(dim=-1)
            preds = [decode_tokens(x) for x in pred_ids]

        train_preds.extend(preds)
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    # Step LR scheduler after each epoch
    scheduler.step()

    train_car = char_accuracy(train_preds, train_labels)
    train_wer_score = wer(train_labels, train_preds) * 100
    avg_train_loss = train_loss / len(train_loader)

    # --- VALIDATE ---
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input  = targets[:, :-1]
            decoder_target = targets[:, 1:]

            logits   = model(images, decoder_input)
            loss     = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))
            val_loss += loss.item()

            pred_ids = model.predict(images, use_beam=False)
            preds    = [decode_tokens(x) for x in pred_ids]

            val_preds.extend(preds)
            val_labels.extend(labels)

    val_car       = char_accuracy(val_preds, val_labels)
    val_wer_score = wer(val_labels, val_preds) * 100
    avg_val_loss  = val_loss / len(val_loader)

    print("\n" + "="*60)
    print(f"EPOCH {epoch}  |  LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"TRAIN LOSS: {avg_train_loss:.4f}  |  VAL LOSS: {avg_val_loss:.4f}")
    print(f"TRAIN CAR:  {train_car:.2f}%      |  VAL CAR:  {val_car:.2f}%")
    print(f"TRAIN WER:  {train_wer_score:.2f}%      |  VAL WER:  {val_wer_score:.2f}%")
    print("="*60)

    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        patience_counter = 0
        torch.save({
            "model":    model.state_dict(),
            "epoch":    epoch,
            "val_loss": avg_val_loss,
        }, CHECKPOINT_PATH)
        print(">>> Checkpoint saved (val loss improved).")
    else:
        patience_counter += 1
        print(f">>> No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOP] Patience exhausted at epoch {epoch}.")
        break


# ============================================================
# FINAL EVALUATION (with beam search)
# ============================================================
print("\nLoading best checkpoint for final evaluation...")
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing with beam search"):
        images = images.to(DEVICE)
        beam_results = model.predict(images, use_beam=True)
        preds        = decode_beam_results(beam_results)
        test_preds.extend(preds)
        test_labels.extend(labels)

test_car       = char_accuracy(test_preds, test_labels)
test_wer_score = wer(test_labels, test_preds) * 100

print("\n" + "#"*40)
print("FINAL TEST METRICS")
print(f"TEST CAR (Character Accuracy Rate) : {test_car:.2f}%")
print(f"TEST WER (Word Error Rate)         : {test_wer_score:.2f}%")
print("#"*40)


# ============================================================
# SAMPLE PREVIEW
# ============================================================
print("\nSAMPLE PREDICTIONS:\n")
for i in range(min(15, len(test_labels))):
    print(f"[{i+1:02d}] TRUE : {test_labels[i]}")
    print(f"     PRED : {test_preds[i]}")
    print("-" * 60)

VOCAB SIZE: 103
Initializing READ dataset extraction...


Parsing XMLs in page: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 27.13it/s]
/tmp/ipykernel_1412115/3660117178.py:218: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 25), p=0.3),



Data breakdown:
TRAIN: 8366
VAL:   728
TEST:  312
Aligning Pretrained Multilingual Token Embedding Weights...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


EPOCH 45  |  LR: 3.12e-05
TRAIN LOSS: 0.0360  |  VAL LOSS: 0.4046
TRAIN CAR:  62.86%      |  VAL CAR:  48.61%
TRAIN WER:  21.76%      |  VAL WER:  47.27%
>>> No improvement. Patience: 19/20


Epoch 46 | Loss: 0.0294: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [02:00<00:00,  8.68it/s]
Validating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:21<00:00,  4.15it/s]



EPOCH 46  |  LR: 3.04e-05
TRAIN LOSS: 0.0353  |  VAL LOSS: 0.4099
TRAIN CAR:  62.89%      |  VAL CAR:  48.19%
TRAIN WER:  21.55%      |  VAL WER:  47.49%
>>> No improvement. Patience: 20/20

[EARLY STOP] Patience exhausted at epoch 46.

Loading best checkpoint for final evaluation...
Loaded checkpoint from epoch 26


Testing with beam search: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 39/39 [09:02<00:00, 13.91s/it]


########################################
FINAL TEST METRICS
TEST CAR (Character Accuracy Rate) : 48.38%
TEST WER (Word Error Rate)         : 48.93%
########################################

SAMPLE PREDICTIONS:

[01] TRUE : Im Andern Mandat.
     PRED : Janndridern Mandt¬
------------------------------------------------------------
[02] TRUE : Pitet vmb die Inwonūng
     PRED : Pitet vmb die Inwonn<UNK>ng
------------------------------------------------------------
[03] TRUE : Schreiben
     PRED : Schreiben
------------------------------------------------------------
[04] TRUE : Mūsicant
     PRED : M<UNK>sicant
------------------------------------------------------------
[05] TRUE : hanns Maximilian Jōrger.
     PRED : Hanns Mainn im Irger
------------------------------------------------------------
[06] TRUE : Loblichen Cammer
     PRED : Lablicher Camen¬
------------------------------------------------------------
[07] TRUE : dergleichen: one habende
     PRED : dergleichen: one ha

## 3

In [ ]:
# ============================================================
# HVLT FOR READ DATASET - FINAL CORRECTED VERSION
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("albumentations not installed. pip install albumentations")


# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Training/page"
VAL_DIR   = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Validation/page"

IMG_HEIGHT = 64
IMG_WIDTH  = 512
BATCH_SIZE = 8
LR         = 5e-5
MAX_EPOCHS = 100
MAX_SEQ_LEN = 200
PATIENCE   = 20          # BUG FIX: was 0
BEAM_WIDTH = 5
LLRD_DECAY = 0.75
ENCODER_UNFREEZE_EPOCH = 5

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED        = 42
USE_AMP     = True
CHECKPOINT_PATH = "best_read_hvlt_4.pt"


# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# VOCABULARY
# BUG FIX: was using DATASET_CHARS (undefined), now READ_CHARS
# ============================================================
READ_CHARS = (
    " !+,-./:0123456789"
    "()<>"
    "ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz"
    "\u00ac"   # ¬
    "\u00be"   # ¾
    "\u00d6"   # Ö
    "\u00df"   # ß
    "\u00e4"   # ä
    "\u00f6"   # ö
    "\u00fc"   # ü
    "\u00ff"   # ÿ
    "\u0101"   # ā
    "\u0113"   # ē
    "\u014d"   # ō
    "\u016b"   # ū
    "\u0233"   # ȳ
    "\u0304"   # combining macron
    "\u0308"   # combining diaeresis
    "\u2014"   # — em dash
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS     = SPECIAL_TOKENS + sorted(set(READ_CHARS))  # BUG FIX: READ_CHARS not DATASET_CHARS

char2idx  = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char  = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print(f"VOCAB SIZE: {VOCAB_SIZE}")


# ============================================================
# PAGE XML PARSING
# BUG FIX: text_content was used before assignment
# ============================================================
def parse_page_xml_dataset(directory_path):
    extracted_samples = []

    if not os.path.exists(directory_path):
        print(f"Warning: Path does not exist -> {directory_path}")
        return extracted_samples

    xml_files = [f for f in os.listdir(directory_path) if f.endswith('.xml')]
    ns = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

    for xml_file in tqdm(xml_files, desc=f"Parsing {os.path.basename(directory_path)}"):
        xml_path = os.path.join(directory_path, xml_file)
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            page_elem = root.find('.//ns:Page', ns)
            if page_elem is None:
                continue

            img_path = os.path.join(directory_path, page_elem.get('imageFilename'))
            if not os.path.exists(img_path):
                continue

            master_img = cv2.imread(img_path)
            if master_img is None:
                continue

            img_h, img_w = master_img.shape[:2]

            for text_line in root.findall('.//ns:TextLine', ns):
                coords_elem = text_line.find('ns:Coords', ns)
                if coords_elem is None:
                    continue
                points_str = coords_elem.get('points')

                unicode_elem = text_line.find('.//ns:Unicode', ns)
                if unicode_elem is None or unicode_elem.text is None:
                    continue

                # BUG FIX: assign text_content FIRST, then normalize, then filter
                text_content = unicode_elem.text.strip()
                if not text_content:
                    continue
                text_content = unicodedata.normalize("NFC", text_content)

                # Skip lines with characters outside vocabulary
                unknown_chars = set(text_content) - set(ALL_TOKENS)
                if unknown_chars:
                    continue

                try:
                    pts = np.array(
                        [list(map(int, pt.split(','))) for pt in points_str.split(' ')],
                        dtype=np.int32
                    )
                    x, y, w, h = cv2.boundingRect(pts)

                    if w < 5 or h < 5 or x < 0 or y < 0 or (x+w) > img_w or (y+h) > img_h:
                        continue

                    line_crop = cv2.cvtColor(master_img[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
                    extracted_samples.append((line_crop, text_content))
                except Exception:
                    continue

        except Exception as e:
            print(f"Skipping {xml_file}: {e}")
            continue

    return extracted_samples


print("Parsing READ dataset...")
train_samples = parse_page_xml_dataset(TRAIN_DIR)
val_samples   = parse_page_xml_dataset(VAL_DIR)

val_samples, test_samples = train_test_split(
    val_samples, test_size=0.3, random_state=SEED
)

print(f"\nData breakdown:")
print(f"TRAIN : {len(train_samples)}")
print(f"VAL   : {len(val_samples)}")
print(f"TEST  : {len(test_samples)}")

if len(train_samples) == 0 or len(val_samples) == 0:
    raise ValueError("No data found. Check directory paths.")

# Verify vocab coverage on full dataset
print("\nVerifying vocab coverage...")
unk_lines = 0
for _, text in train_samples + val_samples:
    bad = [c for c in text if c not in char2idx]
    if bad:
        unk_lines += 1
print(f"Lines with unknown chars: {unk_lines} / {len(train_samples)+len(val_samples)}")
if unk_lines == 0:
    print("Vocab coverage: PERFECT — zero unknown chars")


# ============================================================
# AUGMENTATION
# ============================================================
if HAS_ALBUMENTATIONS:
    strong_aug = A.Compose([
        A.ElasticTransform(alpha=40, sigma=6, p=0.5),
        A.Rotate(limit=3, border_mode=cv2.BORDER_REPLICATE, p=0.4),
        A.GaussNoise(var_limit=(5, 35), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.Morphological(scale=(1, 3), operation='dilation', p=0.25),
        A.Morphological(scale=(1, 3), operation='erosion',  p=0.25),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    ])
else:
    strong_aug = None


def preprocess_cropped_image(cv2_img, augment=False):
    if augment:
        if strong_aug is not None:
            cv2_img = strong_aug(image=cv2_img)["image"]
        else:
            if random.random() < 0.5:
                cv2_img = cv2.convertScaleAbs(cv2_img,
                    alpha=random.uniform(0.8, 1.2), beta=random.randint(-15, 15))
            if random.random() < 0.3:
                k = random.choice([3, 5])
                cv2_img = cv2.GaussianBlur(cv2_img, (k, k), 0)

    h, w  = cv2_img.shape[:2]
    new_w = max(1, int(w * IMG_HEIGHT / h))
    img   = cv2.resize(cv2_img, (new_w, IMG_HEIGHT))

    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.tensor(np.transpose(img, (2, 0, 1)), dtype=torch.float32)


# ============================================================
# TEXT UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)
    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX
    return torch.tensor(tokens, dtype=torch.long)


def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)


def decode_beam_results(beam_results):
    return [decode_tokens(t.tolist()) for t in beam_results]


# ============================================================
# DATASET
# ============================================================
class READLineDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cv2_img, text = self.samples[idx]
        return preprocess_cropped_image(cv2_img, self.augment), encode_text(text), text


train_loader = DataLoader(READLineDataset(train_samples, augment=True),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(READLineDataset(val_samples,   augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(READLineDataset(test_samples,  augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


# ============================================================
# POSITIONAL BRIDGE
# ============================================================
class PositionalBridge(nn.Module):
    def __init__(self, in_channels=768, d_model=512, vis_seq_len=1024):
        super().__init__()
        self.pool      = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj      = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed


# ============================================================
# DECODER
# ============================================================
class READDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        self.token_embed  = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed    = nn.Embedding(MAX_SEQ_LEN, d_model)
        self.embed_dropout = nn.Dropout(0.2)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=2048,
            dropout=0.3, batch_first=True
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Loading XLM-R embeddings...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            n   = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n, :d_model]
            del xlm
            print("XLM-R embeddings loaded.")
        except Exception as e:
            print(f"XLM-R init skipped: {e}")

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x    = self.embed_dropout(self.token_embed(tgt_tokens) + self.pos_embed(pos))

        tgt_mask             = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(tgt=x, memory=memory,
                           tgt_mask=tgt_mask,
                           tgt_key_padding_mask=tgt_key_padding_mask)
        return self.output_proj(out)

    @torch.no_grad()
    def beam_decode(self, memory, beam_width=BEAM_WIDTH, max_len=MAX_SEQ_LEN):
        B, results = memory.shape[0], []
        for b in range(B):
            mem, beams, completed = memory[b:b+1], [(0.0, [SOS_IDX])], []
            for _ in range(max_len):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                        continue
                    t     = torch.tensor([tokens], dtype=torch.long, device=memory.device)
                    log_p = F.log_softmax(self.forward(mem, t)[0, -1], dim=-1)
                    for s, idx in zip(*log_p.topk(beam_width)):
                        candidates.append((score + s.item(), tokens + [idx.item()]))
                if not candidates:
                    break
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:beam_width]
                if all(t[-1] == EOS_IDX for _, t in beams):
                    completed.extend(beams)
                    break
            if not completed:
                completed = beams
            results.append(torch.tensor(
                max(completed, key=lambda x: x[0])[1][1:], dtype=torch.long))
        return results

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B         = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished  = torch.zeros(B, dtype=torch.bool, device=memory.device)
        for _ in range(max_len):
            next_token = self.forward(memory, generated)[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)
            generated  = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished  |= (next_token == EOS_IDX)
            if finished.all():
                break
        return generated[:, 1:]


# ============================================================
# MODEL
# ============================================================
class HVLT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            "swin_small_patch4_window7_224",
            pretrained=True, features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH), strict_img_size=False,
        )
        self.bridge  = PositionalBridge(in_channels=768, d_model=512, vis_seq_len=1024)
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE)

    def forward(self, images, tgt_tokens):
        return self.decoder(self.bridge(self.encoder(images)[-1]), tgt_tokens)

    def _encode(self, images):
        return self.bridge(self.encoder(images)[-1])

    @torch.no_grad()
    def predict(self, images, use_beam=True):
        memory = self._encode(images)
        return self.decoder.beam_decode(memory) if use_beam else self.decoder.greedy_decode(memory)


# ============================================================
# SETUP
# ============================================================
model = HVLT().to(DEVICE)

def set_encoder_trainable(model, trainable):
    for param in model.encoder.parameters():
        param.requires_grad = trainable
    print(f">>> Encoder {'UNFROZEN' if trainable else 'FROZEN'}")

set_encoder_trainable(model, trainable=False)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.15)


def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        correct += sum(p[i] == l[i] for i in range(n))
        total   += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)


# ============================================================
# LLRD — fixed layer name matching for Swin-Small
# ============================================================
def get_llrd_param_groups(model, base_lr, decay):
    groups = {"dec_bridge": [], "stage3": [], "stage2": [], "stage1": [], "stage0": []}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("decoder.") or name.startswith("bridge."):
            groups["dec_bridge"].append(param)
        elif "encoder.layers.3" in name:
            groups["stage3"].append(param)
        elif "encoder.layers.2" in name:
            groups["stage2"].append(param)
        elif "encoder.layers.1" in name:
            groups["stage1"].append(param)
        else:
            groups["stage0"].append(param)

    param_groups = [
        {"params": groups["dec_bridge"], "lr": base_lr},
        {"params": groups["stage3"],     "lr": base_lr * decay},
        {"params": groups["stage2"],     "lr": base_lr * decay**2},
        {"params": groups["stage1"],     "lr": base_lr * decay**3},
        {"params": groups["stage0"],     "lr": base_lr * decay**4},
    ]

    labels = ["decoder+bridge", "swin stage 3", "swin stage 2", "swin stage 1", "swin stage 0+embed"]
    for label, g in zip(labels, param_groups):
        flag = " ⚠ EMPTY" if len(g["params"]) == 0 else ""
        print(f"  {label:25s}  lr={g['lr']:.2e}  ({len(g['params'])} tensors){flag}")

    return param_groups


print("\nBuilding LLRD param groups (encoder frozen):")
param_groups = get_llrd_param_groups(model, LR, LLRD_DECAY)
optimizer    = Adam(param_groups)
scheduler    = CosineAnnealingLR(optimizer, T_max=ENCODER_UNFREEZE_EPOCH, eta_min=1e-7)
scaler       = GradScaler("cuda", enabled=USE_AMP)

print(f"\nTOTAL PARAMS: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


# ============================================================
# TRAINING LOOP
# ============================================================
best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):

    if epoch == ENCODER_UNFREEZE_EPOCH + 1:
        set_encoder_trainable(model, trainable=True)
        param_groups = get_llrd_param_groups(model, LR, LLRD_DECAY)
        optimizer    = Adam(param_groups, weight_decay=1e-4)
        scheduler    = CosineAnnealingLR(optimizer,
                           T_max=MAX_EPOCHS - ENCODER_UNFREEZE_EPOCH, eta_min=1e-7)
        print(">>> Optimizer rebuilt with full LLRD")

    # --- TRAIN ---
    model.train()
    train_loss, train_preds, train_labels = 0, [], []

    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input, decoder_target = targets[:, :-1], targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            logits = model(images, decoder_input)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        with torch.no_grad():
            train_preds.extend([decode_tokens(x) for x in logits.argmax(dim=-1)])
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    scheduler.step()
    avg_train_loss  = train_loss / len(train_loader)
    train_car       = char_accuracy(train_preds, train_labels)
    train_wer_score = wer(train_labels, train_preds) * 100

    # --- VALIDATE ---
    model.eval()
    val_loss, val_preds, val_labels = 0, [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input, decoder_target = targets[:, :-1], targets[:, 1:]

            logits    = model(images, decoder_input)
            val_loss += criterion(logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1)).item()

            val_preds.extend([decode_tokens(x) for x in model.predict(images, use_beam=False)])
            val_labels.extend(labels)

    avg_val_loss  = val_loss / len(val_loader)
    val_car       = char_accuracy(val_preds, val_labels)
    val_wer_score = wer(val_labels, val_preds) * 100

    print("\n" + "="*60)
    print(f"EPOCH {epoch}  |  LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"TRAIN LOSS: {avg_train_loss:.4f}  |  VAL LOSS: {avg_val_loss:.4f}")
    print(f"TRAIN CAR:  {train_car:.2f}%      |  VAL CAR:  {val_car:.2f}%")
    print(f"TRAIN WER:  {train_wer_score:.2f}%      |  VAL WER:  {val_wer_score:.2f}%")
    print("="*60)

    if avg_val_loss < best_val_loss:
        best_val_loss, patience_counter = avg_val_loss, 0
        torch.save({"model": model.state_dict(), "epoch": epoch,
                    "val_loss": avg_val_loss}, CHECKPOINT_PATH)
        print(">>> Checkpoint saved.")
    else:
        patience_counter += 1
        print(f">>> No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOP] Epoch {epoch}.")
        break


# ============================================================
# FINAL EVALUATION
# ============================================================
print("\nLoading best checkpoint...")
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    print(f"Loaded from epoch {checkpoint['epoch']}")

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing"):
        test_preds.extend(decode_beam_results(model.predict(images.to(DEVICE), use_beam=True)))
        test_labels.extend(labels)

test_car       = char_accuracy(test_preds, test_labels)
test_wer_score = wer(test_labels, test_preds) * 100

print("\n" + "#"*40)
print("FINAL TEST METRICS")
print(f"TEST CAR : {test_car:.2f}%")
print(f"TEST WER : {test_wer_score:.2f}%")
print("#"*40)

print("\nSAMPLE PREDICTIONS:\n")
for i in range(min(15, len(test_labels))):
    print(f"[{i+1:02d}] TRUE : {test_labels[i]}")
    print(f"     PRED : {test_preds[i]}")
    print("-" * 60)

VOCAB SIZE: 93
Parsing READ dataset...


Parsing page: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 25.93it/s]
/tmp/ipykernel_1412115/4092455799.py:225: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 35), p=0.4),



Data breakdown:
TRAIN : 8366
VAL   : 728
TEST  : 312

Verifying vocab coverage...
Lines with unknown chars: 0 / 9094
Vocab coverage: PERFECT — zero unknown chars
Loading XLM-R embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Epoch 70 | Loss: 1.1406: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [02:15<00:00,  7.74it/s]
Validating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:35<00:00,  2.58it/s]



EPOCH 70  |  LR: 1.14e-05
TRAIN LOSS: 1.1388  |  VAL LOSS: 1.3925
TRAIN CAR:  98.77%      |  VAL CAR:  75.90%
TRAIN WER:  6.96%      |  VAL WER:  34.24%
>>> No improvement. Patience: 19/20


Epoch 71 | Loss: 1.1792: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1046/1046 [02:15<00:00,  7.74it/s]
Validating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:37<00:00,  2.43it/s]



EPOCH 71  |  LR: 1.07e-05
TRAIN LOSS: 1.1379  |  VAL LOSS: 1.3888
TRAIN CAR:  98.82%      |  VAL CAR:  74.85%
TRAIN WER:  6.53%      |  VAL WER:  34.50%
>>> No improvement. Patience: 20/20

[EARLY STOP] Epoch 71.

Loading best checkpoint...
Loaded from epoch 51


Testing:  28%|█████████████████████████████▉                                                                            | 11/39 [05:32<13:58, 29.96s/it]

## 4

In [1]:
# ============================================================
# HVLT FOR READ DATASET - WITH KNOWLEDGE DISTILLATION ENGINE
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel
from jiwer import wer

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("albumentations not installed. pip install albumentations")


# ============================================================
# CONFIG
# ============================================================
TRAIN_DIR = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Training/page"
VAL_DIR   = "/home/mca/Shweta/handwritten text detection/dataset/PublicData/Validation/page"

IMG_HEIGHT = 64
IMG_WIDTH  = 512
BATCH_SIZE = 8
LR         = 5e-5
MAX_EPOCHS = 100
MAX_SEQ_LEN = 200
PATIENCE   = 20  
BEAM_WIDTH = 5
LLRD_DECAY = 0.75
ENCODER_UNFREEZE_EPOCH = 5

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED        = 42
USE_AMP     = True

CHECKPOINT_PATH = "best_read_hvlt_kd_1.pt"
TEACHER_CHECKPOINT = "best_iam_hvlt_4.pt"  # Path to your pre-trained Swin-Base teacher weights

# Knowledge Distillation Parameters
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.4      # Weight for student ground-truth CrossEntropy loss
KD_BETA = 0.4       # Weight for Soft Logit KLD distillation loss
KD_GAMMA = 0.2      # Weight for Feature Alignment MSE loss


# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# VOCABULARY
# ============================================================
READ_CHARS = (
    " !+,-./:0123456789"
    "()<>"
    "ABCDEFGHIJKLMNOPQRSTUVWYZabcdefghijklmnopqrstuvwxyz"
    "\u00ac"   # ¬
    "\u00be"   # ¾
    "\u00d6"   # Ö
    "\u00df"   # ß
    "\u00e4"   # ä
    "\u00f6"   # ö
    "\u00fc"   # ü
    "\u00ff"   # ÿ
    "\u0101"   # ā
    "\u0113"   # ē
    "\u014d"   # ō
    "\u016b"   # ū
    "\u0233"   # ȳ
    "\u0304"   # combining macron
    "\u0308"   # combining diaeresis
    "\u2014"   # — em dash
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS     = SPECIAL_TOKENS + sorted(set(READ_CHARS))  

char2idx  = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char  = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print(f"VOCAB SIZE: {VOCAB_SIZE}")


# ============================================================
# PAGE XML PARSING
# ============================================================
def parse_page_xml_dataset(directory_path):
    extracted_samples = []

    if not os.path.exists(directory_path):
        print(f"Warning: Path does not exist -> {directory_path}")
        return extracted_samples

    xml_files = [f for f in os.listdir(directory_path) if f.endswith('.xml')]
    ns = {'ns': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}

    for xml_file in tqdm(xml_files, desc=f"Parsing {os.path.basename(directory_path)}"):
        xml_path = os.path.join(directory_path, xml_file)
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            page_elem = root.find('.//ns:Page', ns)
            if page_elem is None: continue

            img_path = os.path.join(directory_path, page_elem.get('imageFilename'))
            if not os.path.exists(img_path): continue

            master_img = cv2.imread(img_path)
            if master_img is None: continue

            img_h, img_w = master_img.shape[:2]

            for text_line in root.findall('.//ns:TextLine', ns):
                coords_elem = text_line.find('ns:Coords', ns)
                if coords_elem is None: continue
                points_str = coords_elem.get('points')

                unicode_elem = text_line.find('.//ns:Unicode', ns)
                if unicode_elem is None or unicode_elem.text is None: continue

                text_content = unicode_elem.text.strip()
                if not text_content: continue
                text_content = unicodedata.normalize("NFC", text_content)

                unknown_chars = set(text_content) - set(ALL_TOKENS)
                if unknown_chars: continue

                try:
                    pts = np.array(
                        [list(map(int, pt.split(','))) for pt in points_str.split(' ')],
                        dtype=np.int32
                    )
                    x, y, w, h = cv2.boundingRect(pts)

                    if w < 5 or h < 5 or x < 0 or y < 0 or (x+w) > img_w or (y+h) > img_h:
                        continue

                    line_crop = cv2.cvtColor(master_img[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
                    extracted_samples.append((line_crop, text_content))
                except Exception:
                    continue

        except Exception as e:
            print(f"Skipping {xml_file}: {e}")
            continue

    return extracted_samples


print("Parsing READ dataset...")
train_samples = parse_page_xml_dataset(TRAIN_DIR)
val_samples   = parse_page_xml_dataset(VAL_DIR)

val_samples, test_samples = train_test_split(val_samples, test_size=0.3, random_state=SEED)

print(f"\nData breakdown:")
print(f"TRAIN : {len(train_samples)}")
print(f"VAL   : {len(val_samples)}")
print(f"TEST  : {len(test_samples)}")

if len(train_samples) == 0 or len(val_samples) == 0:
    raise ValueError("No data found. Check directory paths.")


# ============================================================
# AUGMENTATION
# ============================================================
if HAS_ALBUMENTATIONS:
    strong_aug = A.Compose([
        A.ElasticTransform(alpha=40, sigma=6, p=0.5),
        A.Rotate(limit=3, border_mode=cv2.BORDER_REPLICATE, p=0.4),
        A.GaussNoise(var_limit=(5, 35), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.Morphological(scale=(1, 3), operation='dilation', p=0.25),
        A.Morphological(scale=(1, 3), operation='erosion',  p=0.25),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
    ])
else:
    strong_aug = None


def preprocess_cropped_image(cv2_img, augment=False):
    if augment:
        if strong_aug is not None:
            cv2_img = strong_aug(image=cv2_img)["image"]
        else:
            if random.random() < 0.5:
                cv2_img = cv2.convertScaleAbs(cv2_img, alpha=random.uniform(0.8, 1.2), beta=random.randint(-15, 15))
            if random.random() < 0.3:
                k = random.choice([3, 5])
                cv2_img = cv2.GaussianBlur(cv2_img, (k, k), 0)

    h, w  = cv2_img.shape[:2]
    new_w = max(1, int(w * IMG_HEIGHT / h))
    img   = cv2.resize(cv2_img, (new_w, IMG_HEIGHT))

    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        img = np.concatenate([img, pad], axis=1)
    else:
        img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    return torch.tensor(np.transpose(img, (2, 0, 1)), dtype=torch.float32)


# ============================================================
# TEXT UTILS
# ============================================================
def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)
    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX
    return torch.tensor(tokens, dtype=torch.long)


def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX: break
        if t in [PAD_IDX, SOS_IDX]: continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)


def decode_beam_results(beam_results):
    return [decode_tokens(t.tolist()) for t in beam_results]


# ============================================================
# DATASET
# ============================================================
class READLineDataset(Dataset):
    def __init__(self, sample_list, augment=False):
        self.samples = sample_list
        self.augment = augment

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        cv2_img, text = self.samples[idx]
        return preprocess_cropped_image(cv2_img, self.augment), encode_text(text), text


train_loader = DataLoader(READLineDataset(train_samples, augment=True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(READLineDataset(val_samples,   augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(READLineDataset(test_samples,  augment=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)


# ============================================================
# POSITIONAL BRIDGE
# ============================================================
class PositionalBridge(nn.Module):
    def __init__(self, in_channels=768, d_model=512, vis_seq_len=1024):
        super().__init__()
        self.pool      = nn.AdaptiveAvgPool2d((1, vis_seq_len))
        self.proj      = nn.Linear(in_channels, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, vis_seq_len, d_model) * 0.02)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.pool(x).squeeze(2).permute(0, 2, 1)
        return self.proj(x) + self.pos_embed


# ============================================================
# MODEL MODULES (STUDENT DECODER)
# ============================================================
class READDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=4):
        super().__init__()
        self.token_embed  = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_embed    = nn.Embedding(MAX_SEQ_LEN, d_model)
        self.embed_dropout = nn.Dropout(0.2)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=2048,
            dropout=0.3, batch_first=True
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

        print("Loading XLM-R embeddings...")
        try:
            xlm = XLMRobertaModel.from_pretrained("xlm-roberta-base")
            emb = xlm.embeddings.word_embeddings.weight
            n   = min(emb.shape[0], vocab_size)
            self.token_embed.weight.data[:n] = emb[:n, :d_model]
            del xlm
            print("XLM-R embeddings loaded successfully.")
        except Exception as e:
            print(f"XLM-R init skipped: {e}")

    def forward(self, memory, tgt_tokens):
        B, T = tgt_tokens.shape
        pos  = torch.arange(T, device=tgt_tokens.device).unsqueeze(0).expand(B, -1)
        x    = self.embed_dropout(self.token_embed(tgt_tokens) + self.pos_embed(pos))

        tgt_mask             = torch.triu(torch.ones(T, T, device=tgt_tokens.device), diagonal=1).bool()
        tgt_key_padding_mask = (tgt_tokens == PAD_IDX)

        out = self.decoder(tgt=x, memory=memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask)
        return self.output_proj(out)

    @torch.no_grad()
    def beam_decode(self, memory, beam_width=BEAM_WIDTH, max_len=MAX_SEQ_LEN):
        B, results = memory.shape[0], []
        for b in range(B):
            mem, beams, completed = memory[b:b+1], [(0.0, [SOS_IDX])], []
            for _ in range(max_len):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == EOS_IDX:
                        completed.append((score, tokens))
                        continue
                    t     = torch.tensor([tokens], dtype=torch.long, device=memory.device)
                    log_p = F.log_softmax(self.forward(mem, t)[0, -1], dim=-1)
                    for s, idx in zip(*log_p.topk(beam_width)):
                        candidates.append((score + s.item(), tokens + [idx.item()]))
                if not candidates: break
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = candidates[:beam_width]
                if all(t[-1] == EOS_IDX for _, t in beams):
                    completed.extend(beams)
                    break
            if not completed: completed = beams
            results.append(torch.tensor(max(completed, key=lambda x: x[0])[1][1:], dtype=torch.long))
        return results

    @torch.no_grad()
    def greedy_decode(self, memory, max_len=MAX_SEQ_LEN):
        B         = memory.shape[0]
        generated = torch.full((B, 1), SOS_IDX, device=memory.device, dtype=torch.long)
        finished  = torch.zeros(B, dtype=torch.bool, device=memory.device)
        for _ in range(max_len):
            next_token = self.forward(memory, generated)[:, -1].argmax(dim=-1)
            next_token = next_token.masked_fill(finished, PAD_IDX)
            generated  = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            finished  |= (next_token == EOS_IDX)
            if finished.all(): break
        return generated[:, 1:]


# ============================================================
# COMPREHENSIVE KNOWLEDGE DISTILLATION MODEL WRAPPER
# ============================================================
class StudentHVLT(nn.Module):
    def __init__(self):
        super().__init__()
        # Student model based on Swin-Small configuration
        self.encoder = timm.create_model(
            "swin_small_patch4_window7_224", pretrained=True, features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH), strict_img_size=False
        )
        self.bridge  = PositionalBridge(in_channels=768, d_model=512, vis_seq_len=1024)
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE, d_model=512)

    def forward(self, images, tgt_tokens):
        feats  = self.encoder(images)[-1]
        memory = self.bridge(feats)
        logits = self.decoder(memory, tgt_tokens)
        return logits, feats

    def _encode(self, images):
        return self.bridge(self.encoder(images)[-1])

    @torch.no_grad()
    def predict(self, images, use_beam=True):
        memory = self._encode(images)
        return self.decoder.beam_decode(memory) if use_beam else self.decoder.greedy_decode(memory)


class TeacherHVLT(nn.Module):
    def __init__(self):
        super().__init__()
        # Pre-trained Teacher framework loaded via Swin-Base configuration
        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224", pretrained=False, features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH), strict_img_size=False
        )
        self.bridge  = PositionalBridge(in_channels=1024, d_model=768, vis_seq_len=1024)
        # Teacher vocabulary layout matches internal size dimension rules
        self.decoder = READDecoder(vocab_size=VOCAB_SIZE, d_model=768, n_heads=12, n_layers=6)

    def forward(self, images, tgt_tokens):
        feats  = self.encoder(images)[-1]
        memory = self.bridge(feats)
        return self.decoder(memory, tgt_tokens), feats


# ============================================================
# INITIALIZE STUDENT, TEACHER, AND ALIGNMENT LAYERS
# ============================================================
model = StudentHVLT().to(DEVICE)

teacher_model = TeacherHVLT().to(DEVICE)
print(f"\n>>> Loading Pre-trained Teacher Model Checkpoint from {TEACHER_CHECKPOINT}...")
if os.path.exists(TEACHER_CHECKPOINT):
    try:
        teacher_state = torch.load(TEACHER_CHECKPOINT, map_location=DEVICE)
        # Using strict=False here handles structural projection mismatch safely
        teacher_model.load_state_dict(teacher_state["model"], strict=False)
        print(">>> Teacher weights loaded into baseline engine successfully.")
    except Exception as e:
        print(f"Warning: Could not completely parse teacher states: {e}. Running with initialized base.")
else:
    print("Warning: Teacher file path not found. Running initialization manually.")

teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

# Linear transformation to map student latent channel space (768) to teacher feature space (1024)
feature_projector = nn.Conv2d(in_channels=768, out_channels=1024, kernel_size=1).to(DEVICE)


# ============================================================
# SETUP EXTRA TRAINING PARAMETERS
# ============================================================
def set_encoder_trainable(model, trainable):
    for param in model.encoder.parameters():
        param.requires_grad = trainable
    print(f">>> Student Encoder {'UNFROZEN' if trainable else 'FROZEN'}")

set_encoder_trainable(model, trainable=False)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.15)
mse_loss_fn = nn.MSELoss()


def char_accuracy(preds, labels):
    correct, total = 0, 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        correct += sum(p[i] == l[i] for i in range(n))
        total   += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)


# ============================================================
# LLRD CONFIGURATION
# ============================================================
def get_llrd_param_groups(model, projector, base_lr, decay):
    groups = {"dec_bridge_proj": [], "stage3": [], "stage2": [], "stage1": [], "stage0": []}

    # Add projector tensors safely to the optimization flow
    for param in projector.parameters():
        groups["dec_bridge_proj"].append(param)

    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if name.startswith("decoder.") or name.startswith("bridge."):
            groups["dec_bridge_proj"].append(param)
        elif "encoder.layers.3" in name:
            groups["stage3"].append(param)
        elif "encoder.layers.2" in name:
            groups["stage2"].append(param)
        elif "encoder.layers.1" in name:
            groups["stage1"].append(param)
        else:
            groups["stage0"].append(param)

    param_groups = [
        {"params": groups["dec_bridge_proj"], "lr": base_lr},
        {"params": groups["stage3"],     "lr": base_lr * decay},
        {"params": groups["stage2"],     "lr": base_lr * decay**2},
        {"params": groups["stage1"],     "lr": base_lr * decay**3},
        {"params": groups["stage0"],     "lr": base_lr * decay**4},
    ]
    return param_groups


print("\nBuilding LLRD param groups (Student encoder frozen):")
param_groups = get_llrd_param_groups(model, feature_projector, LR, LLRD_DECAY)
optimizer    = Adam(param_groups)
scheduler    = CosineAnnealingLR(optimizer, T_max=ENCODER_UNFREEZE_EPOCH, eta_min=1e-7)
scaler       = GradScaler("cuda", enabled=USE_AMP)


# ============================================================
# TRAINING LOOP WITH DISTILLATION INFERENCE FLOW
# ============================================================
best_val_loss    = float('inf')
patience_counter = 0

for epoch in range(1, MAX_EPOCHS + 1):

    if epoch == ENCODER_UNFREEZE_EPOCH + 1:
        set_encoder_trainable(model, trainable=True)
        param_groups = get_llrd_param_groups(model, feature_projector, LR, LLRD_DECAY)
        optimizer    = Adam(param_groups, weight_decay=1e-4)
        scheduler    = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - ENCODER_UNFREEZE_EPOCH, eta_min=1e-7)
        print(">>> Optimizer rebuilt with full LLRD and projector tracking.")

    # --- TRAIN ---
    model.train()
    feature_projector.train()
    train_loss, train_preds, train_labels = 0, [], []

    pbar = tqdm(train_loader)
    for images, targets, labels in pbar:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        decoder_input, decoder_target = targets[:, :-1], targets[:, 1:]

        optimizer.zero_grad()
        with autocast("cuda", enabled=USE_AMP):
            # Student Pass
            student_logits, student_feats = model(images, decoder_input)
            
            # Frozen Teacher Pass
            with torch.no_grad():
                teacher_logits, teacher_feats = teacher_model(images, decoder_input)

            # 1. Base Task Hard Cross Entropy Loss
            loss_ce = criterion(student_logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1))

            # 2. Soft-Target Token Probability Distillation Loss (KLD)
            soft_student = F.log_softmax(student_logits / KD_TEMPERATURE, dim=-1)
            soft_teacher = F.softmax(teacher_logits / KD_TEMPERATURE, dim=-1)
            loss_kd = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (KD_TEMPERATURE ** 2)

            # 3. Latent Representation Feature Alignment Loss (MSE)
            # Project student latent features maps to evaluate channel variance against teacher layout
            projected_student_feats = feature_projector(student_feats.permute(0, 3, 1, 2))
            loss_feat = mse_loss_fn(projected_student_feats, teacher_feats.permute(0, 3, 1, 2))

            # Consolidated Balanced Distillation Strategy
            loss = (KD_ALPHA * loss_ce) + (KD_BETA * loss_kd) + (KD_GAMMA * loss_feat)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        with torch.no_grad():
            train_preds.extend([decode_tokens(x) for x in student_logits.argmax(dim=-1)])
        train_labels.extend(labels)
        pbar.set_description(f"Epoch {epoch} | Distill Loss: {loss.item():.4f} (CE: {loss_ce.item():.2f}, KD: {loss_kd.item():.2f})")

    scheduler.step()
    avg_train_loss  = train_loss / len(train_loader)
    train_car       = char_accuracy(train_preds, train_labels)
    train_wer_score = wer(train_labels, train_preds) * 100

    # --- VALIDATE ---
    model.eval()
    feature_projector.eval()
    val_loss, val_preds, val_labels = 0, [], []

    with torch.no_grad():
        for images, targets, labels in tqdm(val_loader, desc="Validating"):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            decoder_input, decoder_target = targets[:, :-1], targets[:, 1:]

            student_logits, _ = model(images, decoder_input)
            val_loss += criterion(student_logits.reshape(-1, VOCAB_SIZE), decoder_target.reshape(-1)).item()

            val_preds.extend([decode_tokens(x) for x in model.predict(images, use_beam=False)])
            val_labels.extend(labels)

    avg_val_loss  = val_loss / len(val_loader)
    val_car       = char_accuracy(val_preds, val_labels)
    val_wer_score = wer(val_labels, val_preds) * 100

    print("\n" + "="*60)
    print(f"EPOCH {epoch}  |  LR: {scheduler.get_last_lr()[0]:.2e}")
    print(f"DISTILL TRAIN LOSS: {avg_train_loss:.4f}  |  VAL TASK CE LOSS: {avg_val_loss:.4f}")
    print(f"STUDENT TRAIN CAR:  {train_car:.2f}%      |  VAL CAR:  {val_car:.2f}%")
    print(f"STUDENT TRAIN WER:  {train_wer_score:.2f}%      |  VAL WER:  {val_wer_score:.2f}%")
    print("="*60)

    if avg_val_loss < best_val_loss:
        best_val_loss, patience_counter = avg_val_loss, 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": avg_val_loss}, CHECKPOINT_PATH)
        print(">>> Best Student Checkpoint saved.")
    else:
        patience_counter += 1
        print(f">>> No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\n[EARLY STOP] Epoch {epoch}.")
        break


# ============================================================
# FINAL STUDENT EVALUATION
# ============================================================
print("\nLoading best distilled student checkpoint...")
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint["model"])
    print(f"Loaded from epoch {checkpoint['epoch']}")

model.eval()
test_preds, test_labels = [], []

with torch.no_grad():
    for images, _, labels in tqdm(test_loader, desc="Testing"):
        test_preds.extend(decode_beam_results(model.predict(images.to(DEVICE), use_beam=True)))
        test_labels.extend(labels)

test_car       = char_accuracy(test_preds, test_labels)
test_wer_score = wer(test_labels, test_preds) * 100

print("\n" + "#"*40)
print("FINAL DISTILLED STUDENT TEST METRICS")
print(f"TEST CAR : {test_car:.2f}%")
print(f"TEST WER : {test_wer_score:.2f}%")
print("#"*40)

print("\nSAMPLE PREDICTIONS:\n")
for i in range(min(15, len(test_labels))):
    print(f"[{i+1:02d}] TRUE : {test_labels[i]}")
    print(f"     PRED : {test_preds[i]}")
    print("-" * 60)

/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


VOCAB SIZE: 93
Parsing READ dataset...


Parsing page: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 24.47it/s]
/tmp/ipykernel_313683/3134641538.py:209: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5, 35), p=0.4),



Data breakdown:
TRAIN : 8366
VAL   : 728
TEST  : 312


/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/home/mca/Shweta/handwritten text detection/sbenv/lib/python3.12/site-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:34

...


EPOCH 99  |  LR: 1.14e-07
DISTILL TRAIN LOSS: 1.8593  |  VAL TASK CE LOSS: 4.0928
STUDENT TRAIN CAR:  3.49%      |  VAL CAR:  1.76%
STUDENT TRAIN WER:  188.86%      |  VAL WER:  104.72%
>>> No improvement. Patience: 12/20


Epoch 100 | Distill Loss: 1.8846 (CE: 4.17, KD: 0.48): 100%|██████████████████████████████████████████████████████| 1046/1046 [03:00<00:00,  5.79it/s]
Validating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:42<00:00,  2.16it/s]



EPOCH 100  |  LR: 1.00e-07
DISTILL TRAIN LOSS: 1.8588  |  VAL TASK CE LOSS: 4.0926
STUDENT TRAIN CAR:  3.52%      |  VAL CAR:  1.78%
STUDENT TRAIN WER:  187.13%      |  VAL WER:  104.89%
>>> No improvement. Patience: 13/20

Loading best distilled student checkpoint...
Loaded from epoch 87


Testing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [10:02<00:00, 15.46s/it]


########################################
FINAL DISTILLED STUDENT TEST METRICS
TEST CAR : 2.44%
TEST WER : 126.25%
########################################

SAMPLE PREDICTIONS:

[01] TRUE : Im Andern Mandat.
     PRED : i5 qzhq5 qxzqxhhqqAziqAzzqhhqhqqh
------------------------------------------------------------
[02] TRUE : Pitet vmb die Inwonūng
     PRED : ist . A. xx Ax hAnnAnnAzAqhBqh.
------------------------------------------------------------
[03] TRUE : Schreiben
     PRED : 55 zzha5qxx
------------------------------------------------------------
[04] TRUE : Mūsicant
     PRED : ist . A5 AxzAzhhqqAAin
------------------------------------------------------------
[05] TRUE : hanns Maximilian Jōrger.
     PRED : i5 qzhq5 hxzqxhhqqAxqAzzqhhqhqqh
------------------------------------------------------------
[06] TRUE : Loblichen Cammer
     PRED : zich. A5 AAzzzen. AnnzAzAzzh
------------------------------------------------------------
[07] TRUE : dergleichen: one habende
     PRED 